# ArcPy Geoprocessing Tools

Clipping a layer to a study area, combining polygon layers, and testing whether points are clustered are all jobs for geoprocessing tools, the same tools you run from the Geoprocessing pane in ArcGIS Pro. ArcPy runs every one of these tools from Python, so a script can run a tool on many datasets, check its results, and pass its outputs to the next tool without a single click.

🎯 What You'll Learn

* Set up ArcPy, the data folder, and an output folder for tool outputs
* Find a tool's syntax and run it with positional and keyword arguments
* Store a tool's Result object and read its outputs and messages
* Run the Clip tool and read the messages from a tool that fails
* Run the Union tool with a list of input datasets
* Run the Average Nearest Neighbor tool and convert its outputs to numbers
* Copy a dataset and run a tool of your choosing on the copy

Sections 1 through 6 each demonstrate how something works. In Section 7, a ✏️ **Practice** gives you a chance to choose and run tools yourself.

### ⚙️ Before you start: select the kernel

The **kernel** is the program that runs the Python code in this notebook. Before running any cells, make sure the notebook is using the ArcGIS Pro Python environment. An **environment** is a copy of Python plus the add-on tools installed with it.

Check the kernel name in the top-right corner of the notebook. It should read `arcgispro-py3`, and may include a Python version, for example:
`arcgispro-py3 (Python 3.13.13)`

If it shows a different name or says **Select Kernel**:

1. Click the kernel name (or **Select Kernel**) in the top-right corner
2. Select `arcgispro-py3`
3. If it is not listed, refer back to `jupyter-notebook-basics.ipynb` in `M1-getting-started-with-python\code-alongs`

---

### 1. Setup: ArcPy and environment settings

Each notebook runs in its own kernel. Variables, imports, and settings from another notebook do not carry over, so every ArcPy notebook starts the same way that every ArcPy script does: import ArcPy, then set the environment settings.

⚠️ **Important**: If the next cell raises an exception, try these steps:

* Check that the kernel for this notebook is set to `arcgispro-py3`.
* Open ArcGIS Pro and sign in to the `uagis` organization account with your NetID and password so that ArcGIS Pro can assign you a Named User license.

If none of these steps fixes the exception, reach out to your instructor with a screenshot of the error message.

**DO NOT** proceed with this notebook until this exception is resolved.

In [ ]:
# Import the ArcPy site package
import arcpy

Most tool in this notebook create an **output dataset**, a new dataset that the tool writes. A few tools, such as Calculate Field, modify the input dataset instead of creating a new one. Before running a tool like that, we will copy the input dataset to the `outputs` folder and run the tool on the copy.

To keep inputs and outputs separate, outputs go in an `outputs` folder inside the working folder you created in `course-data-setup.ipynb`, with one subfolder for each module. The source data in `austin_data` stays unchanged, and you can delete the `outputs` folder at any time and recreate everything in it by rerunning the cells.

⚠️ **Important**: Update the following path to match the location of the working folder you created in `course-data-setup.ipynb`.

In [ ]:
# Store the path to your working folder. Update this path to match the
# working folder you created in course-data-setup.ipynb
working_folder = r"C:\workspace\gist"
print(f"working_folder: {working_folder}")

`os.makedirs()` creates the `m3` folder, and the `outputs` folder above it if it does not exist yet.

In [ ]:
# Import os
import os

# Build the path to the outputs folder for this module
output_folder = os.path.join(working_folder, "outputs", "m3")
print(f"output_folder: {output_folder}")

# Create outputs and outputs\m3 if they do not exist yet
os.makedirs(output_folder, exist_ok=True)
print(f"arcpy.Exists(output_folder): {arcpy.Exists(output_folder)}")

As a review, **environment settings** are settings that apply to every tool and function you run. ArcPy stores them as properties of `arcpy.env`:

* `workspace`: the default location of input and output datasets. Once the workspace is set, you can refer to a dataset in it by its name instead of its full path.
* `overwriteOutput`: when `True`, a tool replaces an output dataset that already exists, so you can rerun a cell without an error.

In this notebook, the workspace is set to the output folder, so an output given as a name only is written there. Input datasets are in the `austin_data` folder, so they use full paths built with `os.path.join()`.

In [ ]:
# Set the workspace to the output folder created above
arcpy.env.workspace = output_folder
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

# Allow tools to replace outputs that already exist
arcpy.env.overwriteOutput = True
print(f"arcpy.env.overwriteOutput: {arcpy.env.overwriteOutput}")

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 2. Toolboxes and tools

Geoprocessing tools are called the same way as other ArcPy functions, but they are organized differently.

In ArcGIS Pro, geoprocessing tools are grouped into **toolboxes**, such as Analysis Tools and Data Management Tools. Each toolbox has a short name called an **alias**, such as `analysis` or `management`. In ArcPy, each toolbox alias works like a module, so you run a tool with dot notation: `arcpy.<toolbox alias>.<tool name>()`. For example, the Buffer tool in the Analysis Tools toolbox is `arcpy.analysis.Buffer()`.

The [ArcGIS Pro tool reference](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/main/arcgis-pro-tool-reference.html) lists every toolbox, its alias, and its tools. It includes a description of how each tool works, its Python syntax, and code samples.

`help()` works on tools the same way it works on functions and classes: it displays the tool's syntax and description.

In the syntax, each parameter is listed in order. Required parameters come first. A parameter inside **curly braces** `{}` is **optional**: if you leave it out, the tool uses its default value. The syntax on each tool's page in the tool reference uses the same curly braces.

In [ ]:
# Display the syntax and description of the Buffer tool
help(arcpy.analysis.Buffer)

The syntax above shows that Buffer has three required parameters: `in_features`, `out_feature_class`, and `buffer_distance_or_field`. The buffer distance can be a **linear unit**, a string with a distance and a unit, such as `"500 Meters"`. Running a tool with positional arguments works the same way as calling a function: each argument is matched to a parameter by its position.

In [ ]:
# Build the path to hospitals.shp in the austin_data folder
hospitals = os.path.join(
    working_folder, "austin_data", "shapefiles", "hospitals.shp"
)

print(f"Hospitals Data: {hospitals}")

# Run Buffer with positional arguments
# The output is a name, so it is written to the workspace (output folder)
arcpy.analysis.Buffer(hospitals, "hospitals_buffer.shp", "500 Meters")

Keyword arguments name each parameter, which makes a tool call easier to read. They also let you set one optional parameter without passing all the optional parameters listed before it. In the Buffer syntax, `dissolve_option` is the third optional parameter, after `line_side` and `line_end_type`. A call can span several lines as long as the lines are inside its parentheses.

In [ ]:
# Run Buffer with keyword arguments. The optional dissolve_option parameter
# is set to "ALL" so that overlapping buffers merge into one feature
arcpy.analysis.Buffer(
    in_features=hospitals,
    out_feature_class="hospitals_buffer_dissolved.shp",
    buffer_distance_or_field="500 Meters",
    dissolve_option="ALL"
)

Older code, including a lot of AI-generated code, writes the toolbox alias after the tool name with an underscore, such as `arcpy.Buffer_analysis()`. It runs the same tool. Recognize this syntax when you see it, but you do not need to write it.

In [ ]:
# Older syntax: tool name, underscore, then toolbox alias
# arcpy.Buffer_analysis(hospitals, "hospitals_buffer_old.shp", "500 Meters")

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 3. Tool results and messages

The tool calls in Section 2 did not store their return value. Every tool that runs successfully returns a **Result object**, an object created from the `Result` class that stores information about the tool run, including its outputs and its messages. You store it in a variable the same way you store any function's return value. For all of its properties and methods, see the [Result class](https://doc.esri.com/en/arcgis-pro/latest/arcpy/classes/result.html) page in the ArcGIS Pro Python reference.

In [ ]:
# Store the Result object returned by Buffer
buffer_result = arcpy.analysis.Buffer(
    hospitals, "hospitals_buffer_1km.shp", "1 Kilometers"
)

Printing a Result object displays its first output, which for Buffer is the full path of the output feature class. The printed value looks like a path, but the variable still stores a Result object, not a string.

In [ ]:
# Printing a Result object displays its first output
print(f"buffer_result: {buffer_result}")
print(f"type(buffer_result): {type(buffer_result)}")

To get an output as a string, use its **index**, its position in the tool's list of outputs, starting at `0`. You can index a Result object the same way you index a list, `buffer_result[0]`, or use the `getOutput()` method, `buffer_result.getOutput(0)`. Both return the same string. Esri's documentation uses both forms, so recognize both.

For more information, see [Use tools in Python](https://doc.esri.com/en/arcgis-pro/latest/arcpy/geoprocessing_and_python/using-tools-in-python.html).

In [ ]:
# getOutput(0) returns the first output, the output feature class path
buffer_output = buffer_result.getOutput(0)
print(f"buffer_output: {buffer_output}")
print(f"type(buffer_output): {type(buffer_output)}")

# Indexing with [0] returns the same string
print(f"buffer_result[0]: {buffer_result[0]}")

A tool's output can be the input to the next tool, which is how scripts chain tools together. Some tools do not create a dataset at all. The **Get Count** tool, in the Data Management Tools toolbox, counts the features in a feature class and returns the count as its output. For its syntax, parameters, and code samples, see the [Get Count tool documentation](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/data-management/get-count.html).

In [ ]:
# Pass the Buffer output to Get Count, and store its Result object
# We will use variable buffer_output from the cell above
count_result = arcpy.management.GetCount(buffer_output)

# count_result[0] returns the count
count_output = count_result[0]
print(f"count_output: {count_output}")
print(f"type(count_output): {type(count_output)}")

Indexing a Result object or calling `getOutput()` returns each output as a string, even a number. To use the count in math or a comparison, convert it with `int()`.

In [ ]:
# Convert the count from a string to an integer
feature_count = int(count_output)
print(f"feature_count: {feature_count}")
print(f"type(feature_count): {type(feature_count)}")

Every tool produces **messages**, the same messages you see in the Geoprocessing pane's details: when the tool started, when it finished, and any warnings or errors. Each message has a **severity**:

* `0`: informative message
* `1`: warning message
* `2`: error message

The `getMessages()` method returns messages as one string. With no argument or with `0`, it returns all messages. With `1`, it returns only warning messages, and with `2`, only error messages. A tool that fails raises an exception instead of returning a Result object, so the error messages of a successful tool run are empty. The `messageCount` property stores the number of messages.

In [ ]:
# Check the number of messages
print(f"messageCount: {count_result.messageCount}")

# No argument returns all messages
print(count_result.getMessages())

# 1 returns only warning messages. The quotes show an empty string
print(f"Warnings: '{count_result.getMessages(1)}'")

# 2 returns only error messages
print(f"Errors: '{count_result.getMessages(2)}'")

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 4. Clip tool

The **Clip** tool, in the Analysis Tools toolbox, keeps the parts of the input features that fall inside the clip features. For its syntax, parameters, and code samples, see the [Clip tool documentation](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/analysis/clip.html).

In [ ]:
# Example 1 - Clip a point layer with a polygon layer

# Build the path to austin_boundary.shp in the austin_data folder
austin_bndry = os.path.join(
    working_folder, "austin_data", "shapefiles", "austin_boundary.shp"
)

print(f"Austin Boundary Data: {austin_bndry}")

# We will use the variable that stores hospitals.shp from Section 2
print(f"Hospitals Data: {hospitals}")

# Clip the hospitals (points) using Austin boundary (polygon)
points_clip_result = arcpy.analysis.Clip(
    hospitals, austin_bndry, "austin_hospitals.shp"
)
print(f"type(points_clip_result): {type(points_clip_result)}")

# points_clip_result[0] returns the first output, the output feature class path
points_clip_output = points_clip_result[0]
print(f"points_clip_output: {points_clip_output}")

Clip only accepts certain combinations of geometry types. Point features can be clipped by points, lines, or polygons, but polygon features can only be clipped by polygons. When a tool fails, it raises an exception instead of returning a Result object, and the rest of the cell does not run.

In [ ]:
# Example 2 - Clip a polygon layer with a point layer
# This cell raises an exception on purpose

# We will use variables hospitals and austin_bndry from Example 1
polygon_clip_result = arcpy.analysis.Clip(
    austin_bndry, hospitals, "austin_bndry_clip.shp"
)
print(f"type(polygon_clip_result): {type(polygon_clip_result)}")

The traceback above ends with the tool's error message. Because the tool failed, `polygon_clip_result` was never created. `arcpy.GetMessages()`, an ArcPy function with a capital `G`, returns the messages from the last tool that ran, whether it succeeded or failed. It takes the same severity argument as the `getMessages()` method.

In [ ]:
# polygon_clip_result was never created, so this raises an error
# print(polygon_clip_result)
# NameError: name 'polygon_clip_result' is not defined

# GetMessages() returns the messages from the last tool that ran
print(arcpy.GetMessages())

# 2 returns only the error messages
print(f"Errors: '{arcpy.GetMessages(2)}'")

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 5. Union tool

The **Union** tool, in the Analysis Tools toolbox, combines two or more polygon datasets into one output. The output keeps every area from every input and the attributes of all inputs. For its syntax, parameters, and code samples, see the [Union tool documentation](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/analysis/union.html).

The tools so far took one input dataset per parameter. Union's `in_features` parameter takes a **list** of datasets instead, so you pass a Python list of paths.

The example below also sets two optional parameters:

* `gaps`: a **gap** is an area that is completely enclosed by polygons but not covered by any of them, such as the hole in a ring of parcels around a park. By default, Union leaves gaps empty. Setting `gaps="NO_GAPS"` creates a feature for each gap.
* `join_attributes`: controls which attribute fields the output keeps. Setting `join_attributes="ONLY_FID"` keeps only the FID fields of the inputs.

In [ ]:
# Build the paths to two polygon datasets in the austin_data folder
parks = os.path.join(
    working_folder, "austin_data", "shapefiles", "parks.shp"
)
print(f"Parks Data: {parks}")

# We will use variable austin_bndry from Section 4
print(f"Austin Boundary Data: {austin_bndry}")

# in_features takes a list of datasets
union_inputs = [parks, austin_bndry]
print(f"union_inputs: {union_inputs}")

# Union the parks and austin boundary polygons. Keep only the FID fields
# so the output attribute table stays short
union_result = arcpy.analysis.Union(
    in_features=union_inputs,
    out_feature_class="parks_austin_union.shp",
    join_attributes="ONLY_FID",
    gaps="NO_GAPS"
)
print(f"type(union_result): {type(union_result)}")

# union_result[0] returns the first output, the output feature class path
union_output = union_result[0]
print(f"union_output: {union_output}")

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 6. Average Nearest Neighbor tool

The **Average Nearest Neighbor** tool, in the Spatial Statistics Tools toolbox (alias `stats`), tests whether a set of points is clustered, dispersed, or random. The **nearest neighbor index** is the observed mean distance divided by the expected mean distance. The **p-value** is the probability that a pattern this far from random appears by chance when the points really are random. A small p-value, commonly below `0.05`, means the pattern is **statistically significant**: it is unlikely to be random. The **z-score** measures how far the observed pattern is from a random pattern, in standard deviations.

For the tool's syntax, parameters, and code samples, see the [Average Nearest Neighbor tool documentation](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/spatial-statistics/average-nearest-neighbor.html).

Like the Get Count tool, this tool does not create an output feature class. Its results are **derived outputs**, values the tool calculates and returns in the Result object. The **Derived output** table in the documentation lists them in index order, so the first row is index `0`. The `outputCount` property of a Result object stores the number of outputs.

In [ ]:
# We will use variable hospitals from Section 2
ann_result = arcpy.stats.AverageNearestNeighbor(
    Input_Feature_Class=hospitals,
    Distance_Method="EUCLIDEAN_DISTANCE",
    Generate_Report="GENERATE_REPORT"
)
print(f"outputCount: {ann_result.outputCount}")

# ann_result[0] returns the nearest neighbor index, not a path
nn_index_output = ann_result[0]
print(f"nn_index_output: {nn_index_output}")
print(f"type(nn_index_output): {type(nn_index_output)}")

# ann_result[5] returns the path of the HTML report
print(f"Report Location: {ann_result[5]}")

As with Get Count in Section 3, each derived output is a string. Convert it with `float()` before using it in math or a comparison.

In [ ]:
# Create variables to store the results as a float
nn_index = float(ann_result[0])
print(f"Nearest neighbor index: {nn_index}")

z_score = float(ann_result[1])
print(f"z-score: {z_score}")

p_value = float(ann_result[2])
print(f"p-value: {p_value}")

# Math works on the converted values: observed / expected = index
expected_distance = float(ann_result[3])
observed_distance = float(ann_result[4])
print(f"observed / expected: {observed_distance / expected_distance}")

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 7. Practice: running tools on your own

In this practice, you will copy a dataset to your output folder, then run a tool of your choosing on the copy. Because the second tool works on a copy, the source data in `austin_data` stays unchanged, so you can choose any tool from the [ArcGIS Pro tool reference](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/main/arcgis-pro-tool-reference.html), including one that modifies its input, such as Add Field.

The **Copy Features** tool, in the Data Management Tools toolbox, copies the features of a dataset to a new feature class. For its syntax, parameters, and code samples, see the [Copy Features tool documentation](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/data-management/copy-features.html).

If your tool raises an exception, print `arcpy.GetMessages()` to read the error, then fix the tool call or choose another tool.

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 7a] Build the path to a dataset of your choosing in the austin_data
# folder, and print it.


# 7b] Run Copy Features to copy your dataset from 7a to the output folder.
# Include your initials in the output name, and store the Result object in
# a new variable.


# 7c] Print the output of your Result object from 7b, and use
# arcpy.Exists() to check whether it exists.


In [ ]:
# 7d] Choose a tool from the ArcGIS Pro tool reference that accepts your
# copy from 7b as an input. Use help() to display its syntax and
# description.


# 7e] Run your tool from 7d with keyword arguments, using the output of
# your Result object from 7b as its input and setting at least one
# optional parameter. Store the Result object in a new variable.


# 7f] Print the number of outputs of your Result object from 7e. Print each
# output by its index, and label each one using the tool's documentation.


# 7g] Print all messages of your Result object from 7e.


💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this notebook, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

*End of notebook.*